# Домашнее задание 5
## Катастрофическое забывание

## Цель:
Проверить влияние fine-tuning на исходную модель.


- Описание/Пошаговая инструкция выполнения домашнего задания:
- Скачать датасет ImageNette: https://github.com/fastai/imagenette (ImageNette это подвыборка из 10 классов датасета ImageNet).
- Взять предобученную на обычном ImageNet модель (например, ResNet18) и заменить число классов на 10.
- Дообучить модель на 10 классах ImageNette и замерить точность (эта точность будет считаться базовой). Можно обучить как всю модель, так и только последний слой.
- Сохранить последний слой на 10 классов (слой классификации).
- Далее еще раз обучить эту модель (полностью, все слои), но на задаче классификации датасета CIFAR10.
- Вернуть оригинальный последний слой модели и проверить качество на ImageNette и сравнить с базовой точностью.
- Заморозить веса и дообучить только последний слой (отключить градиент для всех слоев кроме последнего) на ImageNette и проверить удалось ли добиться исходного качества.
- Сделать выводы.


In [1]:
import os
import time
import random
import urllib.request
import tarfile
import copy

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


Device: cuda
GPU: NVIDIA GeForce RTX 3090


## 1. Скачиваем датасет ImageNette

In [2]:
DATA_DIR = "data"
IMAGENETTE_URL = "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz"
IMAGENETTE_ARCHIVE = os.path.join(DATA_DIR, "imagenette2-160.tgz")
IMAGENETTE_ROOT = os.path.join(DATA_DIR, "imagenette2-160")

os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(IMAGENETTE_ROOT):
    if not os.path.exists(IMAGENETTE_ARCHIVE):
        print("Скачиваем ImageNette (160px)...")
        urllib.request.urlretrieve(IMAGENETTE_URL, IMAGENETTE_ARCHIVE)
    print("Распаковываем архив...")
    with tarfile.open(IMAGENETTE_ARCHIVE) as tar:
        tar.extractall(DATA_DIR)
else:
    print("ImageNette уже скачан и распакован")

print("Содержимое:", os.listdir(IMAGENETTE_ROOT))


Распаковываем архив...


Содержимое: ['.DS_Store', 'noisy_imagenette.csv', 'val', 'train']


In [3]:
IMG_SIZE = 160
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.15)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

imagenette_train = datasets.ImageFolder(os.path.join(IMAGENETTE_ROOT, "train"), transform=train_transform)
imagenette_val = datasets.ImageFolder(os.path.join(IMAGENETTE_ROOT, "val"), transform=val_transform)

BATCH_SIZE = 64
imagenette_train_loader = DataLoader(imagenette_train, batch_size=BATCH_SIZE, shuffle=True,
                                      num_workers=4, pin_memory=True)
imagenette_val_loader = DataLoader(imagenette_val, batch_size=BATCH_SIZE, shuffle=False,
                                    num_workers=4, pin_memory=True)

CLASSES = imagenette_train.classes
NUM_CLASSES = len(CLASSES)
print(f"Классы ImageNette ({NUM_CLASSES}): {CLASSES}")
print(f"Train: {len(imagenette_train)} изображений, Val: {len(imagenette_val)} изображений")


Классы ImageNette (10): ['n01440764', 'n02102040', 'n02979186', 'n03000684', 'n03028079', 'n03394916', 'n03417042', 'n03425413', 'n03445777', 'n03888257']
Train: 9469 изображений, Val: 3925 изображений


## 2. ResNet18, предобученная на ImageNet -> замена головы на 10 классов

In [4]:
def build_resnet18(num_classes, pretrained=True):
    weights = ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.resnet18(weights=weights)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model

model = build_resnet18(NUM_CLASSES, pretrained=True).to(DEVICE)
print(model.fc)
print("Всего параметров:", sum(p.numel() for p in model.parameters()))


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/ubuntu/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


  0%|          | 0.00/44.7M [00:00<?, ?B/s]

  0%|          | 128k/44.7M [00:00<02:41, 289kB/s]

  1%|          | 256k/44.7M [00:00<01:45, 441kB/s]

  1%|          | 384k/44.7M [00:00<01:16, 606kB/s]

...

 99%|█████████▉| 44.1M/44.7M [00:35<00:00, 1.45MB/s]

 99%|█████████▉| 44.4M/44.7M [00:35<00:00, 1.50MB/s]

100%|█████████▉| 44.6M/44.7M [00:35<00:00, 1.54MB/s]

100%|██████████| 44.7M/44.7M [00:35<00:00, 1.32MB/s]

Linear(in_features=512, out_features=10, bias=True)
Всего параметров: 11181642


### Функции обучения и оценки

In [5]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss, running_correct, total = 0.0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_correct += (outputs.argmax(1) == labels).sum().item()
        total += inputs.size(0)
    return running_loss / total, running_correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, running_correct, total = 0.0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * inputs.size(0)
        running_correct += (outputs.argmax(1) == labels).sum().item()
        total += inputs.size(0)
    return running_loss / total, running_correct / total


def fit(model, train_loader, val_loader, optimizer, epochs, device, criterion=None, tag=""):
    criterion = criterion or nn.CrossEntropyLoss()
    history = []
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        dt = time.time() - t0
        print(f"[{tag}] epoch {epoch}/{epochs} "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} ({dt:.1f}s)")
        history.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
                         "val_loss": val_loss, "val_acc": val_acc})
    return history


## 3. Дообучение всей модели на ImageNette (Базовая точность)

In [6]:
EPOCHS_IMAGENETTE_BASELINE = 5
LR = 1e-4

optimizer = optim.Adam(model.parameters(), lr=LR)
history_baseline = fit(model, imagenette_train_loader, imagenette_val_loader, optimizer,
                        EPOCHS_IMAGENETTE_BASELINE, DEVICE, tag="ImageNette-baseline")

baseline_val_loss, baseline_val_acc = evaluate(model, imagenette_val_loader, nn.CrossEntropyLoss(), DEVICE)
print(f"\nБазовая точность на ImageNette (val): {baseline_val_acc:.4f}")


[ImageNette-baseline] epoch 1/5 train_loss=0.3147 train_acc=0.9131 val_loss=0.1503 val_acc=0.9539 (5.5s)


[ImageNette-baseline] epoch 2/5 train_loss=0.0860 train_acc=0.9755 val_loss=0.1313 val_acc=0.9567 (5.2s)


[ImageNette-baseline] epoch 3/5 train_loss=0.0513 train_acc=0.9870 val_loss=0.1612 val_acc=0.9513 (5.2s)


[ImageNette-baseline] epoch 4/5 train_loss=0.0339 train_acc=0.9894 val_loss=0.1402 val_acc=0.9564 (5.2s)


[ImageNette-baseline] epoch 5/5 train_loss=0.0289 train_acc=0.9913 val_loss=0.1607 val_acc=0.9541 (5.3s)



Базовая точность на ImageNette (val): 0.9541


## 4. Сохраняем последний слой (голову классификации на 10 классов ImageNette)

In [7]:
IMAGENETTE_HEAD_PATH = "imagenette_head.pth"
torch.save(model.fc.state_dict(), IMAGENETTE_HEAD_PATH)
print(f"Голова классификации ImageNette сохранена в {IMAGENETTE_HEAD_PATH}")


Голова классификации ImageNette сохранена в imagenette_head.pth


## 5. Дообучение всей модели (все слои) на CIFAR10

In [8]:
cifar_train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

cifar_val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

cifar_train = datasets.CIFAR10(root=DATA_DIR, train=True, download=True, transform=cifar_train_transform)
cifar_test = datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=cifar_val_transform)

CIFAR_BATCH_SIZE = 128
cifar_train_loader = DataLoader(cifar_train, batch_size=CIFAR_BATCH_SIZE, shuffle=True,
                                 num_workers=4, pin_memory=True)
cifar_test_loader = DataLoader(cifar_test, batch_size=CIFAR_BATCH_SIZE, shuffle=False,
                                num_workers=4, pin_memory=True)

print(f"Классы CIFAR10: {cifar_train.classes}")
print(f"Train: {len(cifar_train)}, Test: {len(cifar_test)}")


  0%|          | 0.00/170M [00:00<?, ?B/s]

  0%|          | 32.8k/170M [00:00<29:35, 96.0kB/s]

  0%|          | 65.5k/170M [00:00<19:44, 144kB/s] 

  0%|          | 98.3k/170M [00:00<20:33, 138kB/s]

  0%|          | 131k/170M [00:00<20:51, 136kB/s] 

...

100%|█████████▉| 170M/170M [21:16<00:00, 61.0kB/s]

100%|█████████▉| 170M/170M [21:17<00:00, 65.2kB/s]

100%|██████████| 170M/170M [21:17<00:00, 134kB/s] 

/home/ubuntu/projects/ml/cv/hw_05/.venv/lib/python3.11/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Классы CIFAR10: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Train: 50000, Test: 10000


In [9]:
# Новая голова для CIFAR10 (тоже 10 классов, но другая задача) -- все слои остаются обучаемыми
model.fc = nn.Linear(model.fc.in_features, 10).to(DEVICE)

EPOCHS_CIFAR = 4
optimizer = optim.Adam(model.parameters(), lr=LR)
history_cifar = fit(model, cifar_train_loader, cifar_test_loader, optimizer,
                     EPOCHS_CIFAR, DEVICE, tag="CIFAR10-full")

cifar_test_loss, cifar_test_acc = evaluate(model, cifar_test_loader, nn.CrossEntropyLoss(), DEVICE)
print(f"\nТочность на CIFAR10 (test) после полного дообучения: {cifar_test_acc:.4f}")


[CIFAR10-full] epoch 1/4 train_loss=0.3492 train_acc=0.8851 val_loss=0.2123 val_acc=0.9271 (21.5s)


[CIFAR10-full] epoch 2/4 train_loss=0.1261 train_acc=0.9587 val_loss=0.1682 val_acc=0.9418 (21.6s)


[CIFAR10-full] epoch 3/4 train_loss=0.0708 train_acc=0.9770 val_loss=0.1830 val_acc=0.9399 (21.9s)


[CIFAR10-full] epoch 4/4 train_loss=0.0464 train_acc=0.9859 val_loss=0.1737 val_acc=0.9440 (22.1s)



Точность на CIFAR10 (test) после полного дообучения: 0.9440


## 6. Возвращаем оригинальную голову ImageNette и проверяем качество

Backbone теперь обучен под CIFAR10. Возвращаем сохранённую голову ImageNette (10 классов ImageNette) и проверяем, что стало с качеством на ImageNette.

In [10]:
imagenette_head_state = torch.load(IMAGENETTE_HEAD_PATH, map_location=DEVICE)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES).to(DEVICE)
model.fc.load_state_dict(imagenette_head_state)

restored_val_loss, restored_val_acc = evaluate(model, imagenette_val_loader, nn.CrossEntropyLoss(), DEVICE)
print(f"Точность на ImageNette (val) с backbone, обученным на CIFAR10, "
      f"и оригинальной головой ImageNette: {restored_val_acc:.4f}")
print(f"Базовая точность (до обучения на CIFAR10): {baseline_val_acc:.4f}")
print(f"Разница: {restored_val_acc - baseline_val_acc:+.4f}")


Точность на ImageNette (val) с backbone, обученным на CIFAR10, и оригинальной головой ImageNette: 0.5893
Базовая точность (до обучения на CIFAR10):                        0.9541
Разница:                                                          -0.3648


## 7. Замораживаем backbone и дообучаем только последний слой на ImageNette

Отключаем градиент для всех слоёв, кроме `fc`, и дообучаем только голову (уже восстановленную на шаге 6) поверх "смещённого" backbone. Проверяем, удаётся ли вернуться к базовому качеству.

In [11]:
for name, param in model.named_parameters():
    param.requires_grad = name.startswith("fc.")

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f"Обучаемых параметров: {n_trainable} из {n_total} ({100 * n_trainable / n_total:.2f}%)")

EPOCHS_LINEAR_PROBE = 8
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
history_linear_probe = fit(model, imagenette_train_loader, imagenette_val_loader, optimizer,
                            EPOCHS_LINEAR_PROBE, DEVICE, tag="ImageNette-linear-probe")

probe_val_loss, probe_val_acc = evaluate(model, imagenette_val_loader, nn.CrossEntropyLoss(), DEVICE)
print(f"\nТочность после дообучения только последнего слоя: {probe_val_acc:.4f}")


Обучаемых параметров: 5130 из 11181642 (0.05%)


[ImageNette-linear-probe] epoch 1/8 train_loss=0.1427 train_acc=0.9617 val_loss=0.2145 val_acc=0.9332 (3.9s)


[ImageNette-linear-probe] epoch 2/8 train_loss=0.1008 train_acc=0.9712 val_loss=0.2051 val_acc=0.9343 (3.9s)


[ImageNette-linear-probe] epoch 3/8 train_loss=0.0827 train_acc=0.9766 val_loss=0.2054 val_acc=0.9383 (3.9s)


[ImageNette-linear-probe] epoch 4/8 train_loss=0.0774 train_acc=0.9757 val_loss=0.2106 val_acc=0.9394 (3.9s)


[ImageNette-linear-probe] epoch 5/8 train_loss=0.0695 train_acc=0.9778 val_loss=0.2062 val_acc=0.9396 (3.9s)


[ImageNette-linear-probe] epoch 6/8 train_loss=0.0654 train_acc=0.9786 val_loss=0.1986 val_acc=0.9439 (3.9s)


[ImageNette-linear-probe] epoch 7/8 train_loss=0.0652 train_acc=0.9806 val_loss=0.2148 val_acc=0.9376 (3.9s)


[ImageNette-linear-probe] epoch 8/8 train_loss=0.0630 train_acc=0.9797 val_loss=0.2056 val_acc=0.9394 (3.9s)



Точность после дообучения только последнего слоя: 0.9394


In [12]:
print("Сводка точности на ImageNette (val):")
print(f"  1) Базовая (вся модель дообучена на ImageNette):                         {baseline_val_acc:.4f}")
print(f"  2) Backbone дообучен на CIFAR10 + оригинальная голова (без дообучения):  {restored_val_acc:.4f}")
print(f"  3) Backbone заморожен, дообучена только голова на ImageNette:            {probe_val_acc:.4f}")
print()
print(f"Достигли базового качества после дообучения только последнего слоя: "
      f"{'ДА' if probe_val_acc >= baseline_val_acc else 'НЕТ'}")


Сводка точности на ImageNette (val):
  1) Базовая (вся модель дообучена на ImageNette):                         0.9541
  2) Backbone дообучен на CIFAR10 + оригинальная голова (без дообучения):  0.5893
  3) Backbone заморожен, дообучена только голова на ImageNette:            0.9394

Достигли базового качества после дообучения только последнего слоя: НЕТ


## 8. Выводы

Результаты конкретного запуска (точность на ImageNette val, 10 классов):

| Этап | Точность |
|---|---|
| 1) Базовая: вся модель (backbone + голова) дообучена на ImageNette | **0.9541** |
| 2) Backbone дообучен на CIFAR10 (все слои), голова возвращена оригинальная (ImageNette), без дообучения | **0.5893** |
| 3) Backbone заморожен (веса из CIFAR10), дообучена только голова `fc` на ImageNette | **0.9394** |

1. **Базовая точность.** Полное дообучение предобученной на ImageNet ResNet18 (все слои разморожены) на ImageNette даёт точность — 95.4%.

2. **Дообучение на CIFAR10 сильно изменяет признаки под ImageNette.** После того как все слои модели (включая backbone) переобучаются на CIFAR10 (тоже 10 классов, но других), веса сильно смещаются под новую задачу — сама модель на CIFAR10 достигает 94.4% точности. Но если вернуть на место старую голову ImageNette (обученную на признаках до CIFAR10), точность на ImageNette падает с 95.4% до 58.9% из-за катастрофического забывания.

3. **Дообучение только последнего слоя восстанавливает почти всё качество.** Заморозив backbone (уже переобученный на CIFAR10) и дообучив только голову на ImageNette, точность поднимается обратно до 93.9%. Видимо признаки ResNet18 после дообучения на CIFAR10 всё ещё достаточно универсальны.

4. **Практический вывод.**:
   - полное дообучение всех слоёв под новую задачу — самый быстрый способ получить максимальное качество на новой задаче, но ценой этого может быть значительная потеря качества на предыдущих задачах (катастрофическое забывание)
   - обучение только последнего слоя поверх уже смещённого backbone — дешёвый по числу обучаемых параметров способ частично восстановить качество, но он не гарантирует возврата к исходному уровню;
   - если важно одинаково хорошо решать обе задачи одновременно, можно хранить отдельные копии backbone/головы под каждую задачу
